In [0]:
-- DQ check 1: all 8 source tables ingested into bronze (critical; missing table raises and stops pipeline)
CREATE OR REPLACE TEMP VIEW dq_bronze_ingestion AS
WITH bronze_counts AS (
  SELECT 'region' AS table_name, COUNT(*) AS row_count FROM IDENTIFIER(:catalog || '.bronze.region')
  UNION ALL SELECT 'nation', COUNT(*) FROM IDENTIFIER(:catalog || '.bronze.nation')
  UNION ALL SELECT 'airline', COUNT(*) FROM IDENTIFIER(:catalog || '.bronze.airline')
  UNION ALL SELECT 'route', COUNT(*) FROM IDENTIFIER(:catalog || '.bronze.route')
  UNION ALL SELECT 'airline_route', COUNT(*) FROM IDENTIFIER(:catalog || '.bronze.airline_route')
  UNION ALL SELECT 'passenger', COUNT(*) FROM IDENTIFIER(:catalog || '.bronze.passenger')
  UNION ALL SELECT 'booking', COUNT(*) FROM IDENTIFIER(:catalog || '.bronze.booking')
  UNION ALL SELECT 'flight_booking', COUNT(*) FROM IDENTIFIER(:catalog || '.bronze.flight_booking')
)
SELECT
  'BRONZE_INGESTION' AS check_code,
  'all 8 source tables ingested into bronze' AS check_description,
  'critical' AS severity,
  SUM(CASE WHEN row_count = 0 THEN 1 ELSE 0 END) AS failing_rows,
  CASE WHEN SUM(CASE WHEN row_count = 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM bronze_counts

In [0]:
-- DQ check 2: silver primary keys populated and unique (critical)
CREATE OR REPLACE TEMP VIEW dq_silver_pk AS
WITH pk AS (
  SELECT 'region' AS table_name,
    COUNT(*) - COUNT(r_regionkey) AS null_keys,
    COUNT(*) - COUNT(DISTINCT r_regionkey) AS dup_keys
  FROM IDENTIFIER(:catalog || '.silver.region')
  UNION ALL SELECT 'nation', COUNT(*) - COUNT(n_nationkey), COUNT(*) - COUNT(DISTINCT n_nationkey) FROM IDENTIFIER(:catalog || '.silver.nation')
  UNION ALL SELECT 'airline', COUNT(*) - COUNT(al_airlineid), COUNT(*) - COUNT(DISTINCT al_airlineid) FROM IDENTIFIER(:catalog || '.silver.airline')
  UNION ALL SELECT 'route', COUNT(*) - COUNT(ro_routeid), COUNT(*) - COUNT(DISTINCT ro_routeid) FROM IDENTIFIER(:catalog || '.silver.route')
  UNION ALL SELECT 'airline_route', COUNT(*) - COUNT(ar_airlineid, ar_routeid), COUNT(*) - COUNT(DISTINCT ar_airlineid, ar_routeid) FROM IDENTIFIER(:catalog || '.silver.airline_route')
  UNION ALL SELECT 'passenger', COUNT(*) - COUNT(pa_passengerid), COUNT(*) - COUNT(DISTINCT pa_passengerid) FROM IDENTIFIER(:catalog || '.silver.passenger')
  UNION ALL SELECT 'booking', COUNT(*) - COUNT(bk_bookingid), COUNT(*) - COUNT(DISTINCT bk_bookingid) FROM IDENTIFIER(:catalog || '.silver.booking')
  UNION ALL SELECT 'flight_booking', COUNT(*) - COUNT(fb_bookingid, fb_linenumber), COUNT(*) - COUNT(DISTINCT fb_bookingid, fb_linenumber) FROM IDENTIFIER(:catalog || '.silver.flight_booking')
)
SELECT 'SILVER_PK_INTEGRITY', 'silver PKs populated and unique', 'critical',
  SUM(null_keys + dup_keys),
  CASE WHEN SUM(null_keys + dup_keys) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM pk

In [0]:
-- DQ check 3: fact has no null surrogate/natural keys (critical)
CREATE OR REPLACE TEMP VIEW dq_fact_null_keys AS
SELECT 'FACT_NULL_KEYS', 'fact has no null surrogate keys', 'critical',
  COUNT(*),
  CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.fact_bookings')
WHERE passenger_key IS NULL OR route_key IS NULL OR airline_key IS NULL
  OR passenger_id IS NULL OR route_id IS NULL OR airline_id IS NULL

In [0]:
-- DQ check 4: business rules - ticket_price > 0, discount in [0,1], seats >= 1 (critical)
CREATE OR REPLACE TEMP VIEW dq_fact_measures AS
SELECT 'FACT_MEASURE_RULES', 'ticket_price > 0 and discount between 0 and 1 and seats >= 1', 'critical',
  COUNT(*),
  CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM IDENTIFIER(:catalog || '.gold.fact_bookings')
WHERE ticket_price <= 0 OR discount < 0 OR discount > 1 OR seats < 1

In [0]:
-- DQ check 5: gold vs silver row count and net revenue within 0.01% (critical)
CREATE OR REPLACE TEMP VIEW dq_reconciliation AS
WITH gold_tot AS (
  SELECT COUNT(*) AS row_count, SUM(net_revenue) AS revenue
  FROM IDENTIFIER(:catalog || '.gold.fact_bookings')
),
silver_tot AS (
  SELECT COUNT(*) AS row_count, SUM(fb_ticketprice * (1 - fb_discount)) AS revenue
  FROM IDENTIFIER(:catalog || '.silver.flight_booking')
)
SELECT 'GOLD_SILVER_RECONCILIATION', 'gold rows and revenue match silver within 0.01%', 'critical',
  CASE WHEN ABS(g.row_count - s.row_count) / s.row_count > 0.0001
        OR ABS(g.revenue - s.revenue) / s.revenue > 0.0001 THEN 1 ELSE 0 END,
  CASE WHEN ABS(g.row_count - s.row_count) / s.row_count <= 0.0001
        AND ABS(g.revenue - s.revenue) / s.revenue <= 0.0001 THEN 'PASS' ELSE 'FAIL' END
FROM gold_tot g, silver_tot s

In [0]:
-- DQ report: rejected-record counts per table (informational, never fails the pipeline)
CREATE OR REPLACE TEMP VIEW dq_rejected AS
SELECT 'REJECTED_RECORDS' AS check_code, 'region' AS check_description, 'report' AS severity, COUNT(*) AS failing_rows, 'REPORT' AS status FROM IDENTIFIER(:catalog || '.silver.region_quarantine')
UNION ALL SELECT 'REJECTED_RECORDS', 'nation', 'report', COUNT(*), 'REPORT' FROM IDENTIFIER(:catalog || '.silver.nation_quarantine')
UNION ALL SELECT 'REJECTED_RECORDS', 'airline', 'report', COUNT(*), 'REPORT' FROM IDENTIFIER(:catalog || '.silver.airline_quarantine')
UNION ALL SELECT 'REJECTED_RECORDS', 'route', 'report', COUNT(*), 'REPORT' FROM IDENTIFIER(:catalog || '.silver.route_quarantine')
UNION ALL SELECT 'REJECTED_RECORDS', 'airline_route', 'report', COUNT(*), 'REPORT' FROM IDENTIFIER(:catalog || '.silver.airline_route_quarantine')
UNION ALL SELECT 'REJECTED_RECORDS', 'passenger', 'report', COUNT(*), 'REPORT' FROM IDENTIFIER(:catalog || '.silver.passenger_quarantine')
UNION ALL SELECT 'REJECTED_RECORDS', 'booking', 'report', COUNT(*), 'REPORT' FROM IDENTIFIER(:catalog || '.silver.booking_quarantine')
UNION ALL SELECT 'REJECTED_RECORDS', 'flight_booking', 'report', COUNT(*), 'REPORT' FROM IDENTIFIER(:catalog || '.silver.flight_booking_quarantine')

In [0]:
-- combined DQ report (session-temp only, nothing persisted)
CREATE OR REPLACE TEMP VIEW dq_report AS
SELECT * FROM dq_bronze_ingestion
UNION ALL SELECT * FROM dq_silver_pk
UNION ALL SELECT * FROM dq_fact_null_keys
UNION ALL SELECT * FROM dq_fact_measures
UNION ALL SELECT * FROM dq_reconciliation
UNION ALL SELECT * FROM dq_rejected;

-- full DQ report
SELECT * FROM dq_report
ORDER BY severity, check_code, check_description;

-- raise error if any critical check failed (stops the pipeline)
SELECT assert_true(
  (SELECT COUNT(*) FROM dq_report
   WHERE severity = 'critical' AND status = 'FAIL') = 0,
  'DATA QUALITY FAILED: critical check(s) violated'
) AS pipeline_gate